In [1]:
import json
from pathlib import Path
import sys

import numpy as np
import matplotlib.pyplot as plt
%matplotlib widget

from spectra.process_absorber_spectrum import process_spectrum_data, read_spectrum_config_json
from spectra.utilities import read_OE65PRO_spectrum_data_file, find_index_of_nearest

# Prepare small data files

In [2]:
raw_data_directory = Path(".").resolve().parent.parent / "spectra" / "data" / "absorbers" / "avobenzone_in_PEGDA_2020-01-20" / "raw_data"
raw_data_directory

PosixPath('/Users/nordin/Documents/Projects/3D_printer_spectra_repos/3dprinter_spectra_and_tools/spectra/data/absorbers/avobenzone_in_PEGDA_2020-01-20/raw_data')

## Read data

In [3]:
solvent_spectrum = np.loadtxt(raw_data_directory / "QEPB00791_17-10-07-458.txt", skiprows=14)
absorber_spectrum = np.loadtxt(raw_data_directory / "QEPB00791_17-18-57-140.txt", skiprows=14)
absorber_spectrum

array([[ 1.988640e+02,  2.550000e+01],
       [ 1.996710e+02,  2.850000e+01],
       [ 2.004770e+02,  2.850000e+01],
       ...,
       [ 9.994170e+02, -1.500000e+00],
       [ 1.000147e+03, -1.500000e+00],
       [ 1.000877e+03,  5.000000e-01]])

## Reduce data size

In [4]:
stride = 20

reduced_solvent_spectrum = solvent_spectrum[::stride]
reduced_absorber_spectrum = absorber_spectrum[::stride]

print(absorber_spectrum.shape)
print(reduced_absorber_spectrum.shape)

(1044, 2)
(53, 2)


## Plot absorber spectrum

In [5]:
fig, ax = plt.subplots()
ax.plot(absorber_spectrum[:,0], absorber_spectrum[:,1])
ax.plot(reduced_absorber_spectrum[:,0], reduced_absorber_spectrum[:,1])
ax.axvline(300, c='k')
ax.axvline(500, c='k')
ax.set_xlabel('Wavelength (nm)')
ax.set_ylabel('Transmission')
ax.grid();

Canvas(toolbar=Toolbar(toolitems=[('Home', 'Reset original view', 'home', 'home'), ('Back', 'Back to previous …

# Create files

## Reduced data files

In [5]:
write_directory = Path(".").resolve().parent / "files_for_tests" / "data_for_test_process_spectrum_data" 

In [7]:
np.savetxt(write_directory / "raw_data" / "solvent.txt", reduced_solvent_spectrum, delimiter=' ')
np.savetxt(write_directory / "raw_data" / "absorber.txt", reduced_absorber_spectrum, delimiter=' ')

Now manually add 14 lines of header to each reduced raw data file using BBedit, copied from original raw data files.

## spectrum_config.json

In [8]:
spectrum_config = {
    'Measurement': 'Avobenzone absorption in PEGDA',
    'Measurement Date': '2020-01-20',
    'Measurement Spectrometer': 'Ocean Optics FIRE',
    'Absorber': 'Avobenzone',
    'Concentration w/w percent': 0.24,
    'Film thickness microns': 60,
    'Molar mass g/mole': 310.39,
    'Solvent': 'PEGDA',
    'Solvent density g/L': 1110,
    'Solvent absorption measurement file': 'solvent.txt',
    'Solvent + absorber absorption measurement file': 'absorber.txt'
}

with (write_directory / "spectrum_config.json").open('w') as f:
    json.dump(spectrum_config, f)

# Run process_spectrum_data

In [6]:
spectrum_config = read_spectrum_config_json(write_directory / "spectrum_config.json")

process_spectrum_data(write_directory, spectrum_config, wavelength_range=[300,500], num_samples_for_average=3)

Manually move the 2 created files, `absorbance.csv` and `molar_absorptivity.csv` into directory `results`.

Plot results

In [7]:
temp_absorbance = np.loadtxt(write_directory / "results" / "absorbance.csv", delimiter=',')
temp_molar_absorptivity = np.loadtxt(write_directory / "results" / "molar_absorptivity.csv", delimiter=',')

In [8]:
fig, ax = plt.subplots()
ax.plot(temp_absorbance[:,0], temp_absorbance[:,1])
ax.set_xlabel('Wavelength (nm)')
ax.set_ylabel('Absorbance')
ax.grid();

Canvas(toolbar=Toolbar(toolitems=[('Home', 'Reset original view', 'home', 'home'), ('Back', 'Back to previous …

In [9]:
fig, ax = plt.subplots()
ax.plot(temp_molar_absorptivity[:,0], temp_molar_absorptivity[:,1])
ax.set_xlabel('Wavelength (nm)')
ax.set_ylabel('Absorbance')
ax.grid();

Canvas(toolbar=Toolbar(toolitems=[('Home', 'Reset original view', 'home', 'home'), ('Back', 'Back to previous …

In [10]:
temp_spectrum_config = spectrum_config.copy()
temp_spectrum_config

{'Measurement': 'Avobenzone absorption in PEGDA',
 'Measurement Date': '2020-01-20',
 'Measurement Spectrometer': 'Ocean Optics FIRE',
 'Absorber': 'Avobenzone',
 'Concentration w/w percent': 0.24,
 'Film thickness microns': 60,
 'Molar mass g/mole': 310.39,
 'Solvent': 'PEGDA',
 'Solvent density g/L': 1110,
 'Solvent absorption measurement file': 'solvent.txt',
 'Solvent + absorber absorption measurement file': 'absorber.txt'}

In [11]:
del temp_spectrum_config['Solvent absorption measurement file'] 
temp_spectrum_config

{'Measurement': 'Avobenzone absorption in PEGDA',
 'Measurement Date': '2020-01-20',
 'Measurement Spectrometer': 'Ocean Optics FIRE',
 'Absorber': 'Avobenzone',
 'Concentration w/w percent': 0.24,
 'Film thickness microns': 60,
 'Molar mass g/mole': 310.39,
 'Solvent': 'PEGDA',
 'Solvent density g/L': 1110,
 'Solvent + absorber absorption measurement file': 'absorber.txt'}